# 88장 · 인수 기준과 작업 분해

수용 조건은 팀원이 결과를 보고 충족 여부를 판단할 수 있어야 합니다. 출처 하나를 표시한다는 요구를 필요한 필드와 검증 작업으로 나눕니다.

팀의 ‘출처 표시’ 요구를 문서 ID·쪽·절·개정일·판본으로 구체화합니다. 다른 팀원이 같은 응답으로 구현 완료 여부를 판정할 수 있도록 수용 기준을 정합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 팀별 변형을 시도한 뒤 해당 셀 아래에서 예시를 펼쳐 비교할 수 있습니다.


## 시나리오를 검수할 출처 계약 정하기

87장에서 답할 수 있다고 정한 행에는 표시할 원문 위치가 필요합니다. 보류하거나 되묻는 행에는 출처 카드가 비어 있어야 합니다.

이 프로젝트의 노트북은 한 폴더에서 열어 주세요. 첫 셀은 `team-project/final/chapter87.json`을 먼저 찾습니다. 팀 파일이 없으면 [이 장의 시작 예시](chapter88-start.json)를 노트북 옆에 내려받으세요. 출력 첫 줄에서 어느 파일을 읽었는지 확인할 수 있습니다.

In [ ]:
from pathlib import Path
import json
import re

here = Path.cwd()
project_root = here.parent if re.fullmatch(r"chapter[0-9]+", here.name) else here
team_file = project_root / "team-project/final/chapter87.json"
sample_file = here / "chapter88-start.json"
input_file = team_file if team_file.is_file() else sample_file
if not input_file.is_file():
    raise FileNotFoundError("이 장의 시작 예시를 내려받아 노트북 옆에 두세요.")
incoming = json.loads(input_file.read_text(encoding="utf-8"))
print("팀 기록" if input_file == team_file else "교재 예시", input_file.name)
print(incoming['scenarios'])

교재 예시 chapter88-start.json
[{'id': 'Q01', 'q': '장비 A의 점검 전 확인 사항은?', 'status': 'answered', 'source_id': 'RULE-A'}, {'id': 'Q02', 'q': '점검 전 확인 사항은?', 'status': 'need_clarification', 'source_id': None}, {'id': 'Q03', 'q': '장비 A를 다음 달에 교체하나요?', 'status': 'no_evidence', 'source_id': None}]


두 번째 줄은 읽은 파일의 `scenarios` 값입니다.

## 출처 필드의 수용 조건

범위 합의 예시　SC-01을 필수로 정하고 CR-01에서는 답에 전원 표시등이 들어가는지, CR-02에서는 인용한 ID·자료명·페이지·절·개정일·버전이 원문과 맞는지 봅니다. 학생이 출처를 보고 실제 절을 찾을 수 있는지는 동료가 확인합니다. 예약·결제와 점검 승인은 첫 버전에서 제외하고 말투 선택은 여유가 생기면 검토합니다.


### 셀 01 · 수용 기준의 누락 필드

첫 시안이 문서 ID만 보낸다고 가정하세요. 원문을 열려는 사람이 더 필요로 하는 필드를 확인합니다.

자료 담당이 규정 개정일을 찾지 못했다면 출처 검수는 보류합니다. 화면 담당은 임시 자료로 카드 배치를 진행할 수 있습니다. 일정표에 다시 만날 시점과 누가 개정일을 확인해 넘길지 적으세요.


In [1]:
required = ("id", "source", "page", "section", "revised_on", "version")
first_citation = {"id": "RULE-A"}
print([name for name in required if name not in first_citation])

['source', 'page', 'section', 'revised_on', 'version']


source부터 version까지 다섯 이름이 남습니다. 확인자가 문서를 찾게 하려면 화면 담당자만 수정해서는 부족합니다. 자료 담당자가 원문에서 그 값을 확보해야 합니다.

답변 본문이 맞아도 원문을 다시 찾을 위치와 적용 판본이 빠지면 인수 기준을 충족하지 못합니다. 특히 개정 전후 내용이 다른 규정이라면 문서 제목만으로 판본을 정할 수 없습니다. 출력의 누락 목록에서 팀이 실제 화면에 표시할 항목을 정합니다.


#### 출처가 보인다는 수용 기준

<mark>검수자가 같은 원문을 다시 찾을 수 있게 필요한 필드를 정합니다.</mark> 파일 이름만 보여 준 화면과 쪽·절·판본까지 있는 화면으로 같은 규정을 찾아보세요. 팀의 자료에서 실제로 위치를 결정해 준 항목을 인수 기준에 남깁니다.

선수 개념 · 86장 서비스 문제 정의와 최소 시연 범위

## 기능 단위 일정과 자료 준비

일정 예시　월요일에는 기획·자료 담당자가 Q01의 문장과 원문 절을 합의합니다. 화요일에는 자료 담당자가 가상 규정 A·B에 개정일을 적고, 수요일에는 구현 담당자가 입력부터 화면까지 연결합니다. 목요일에는 서로 역할을 바꿔 답을 보고 원문을 찾습니다. 금요일에는 누락된 절 정보와 일정 질문 오답을 따로 기록해 우선순위를 정합니다. 실제 날짜와 담당자는 팀 일정에 맞게 바꾸세요.


### 팀 참고 예시 · 원문 기준 인수 검사


In [2]:
received = {'id': 'RULE-A',
 'equipment': '장비 A',
 'text': '점검 전 전원 표시등을 확인합니다.',
 'source': '가상 장비실 규정 A',
 'page': 1,
 'section': '2.1',
 'revised_on': '2026-11-01',
 'version': 'rules-v1',
 'groups': ['public', 'admin']}
print([name for name in required if name not in received])

[]


빈 목록이 나옵니다. 필드 이름은 모두 있지만 페이지나 절의 오타까지 찾아낸 것은 아닙니다.


## 누락된 출처와 잘못 채운 출처를 따로 검수하기

CR-02의 필드 여섯 개를 모두 채웠는데도 동료가 RULE-A의 근거를 찾지 못했습니다. 비어 있는 항목을 세는 검사와 값이 원문을 가리키는지 대조하는 검사를 따로 해 봅니다.

### 셀 04 · 필드가 모두 있는 잘못된 카드

원문 레코드 received를 복사해 카드의 페이지만 2로 바꿉니다. 문서 담당자가 갖고 있는 원본은 그대로 둡니다.

In [3]:
card = {name: received[name] for name in required}
card["page"] = 2
missing = [name for name in required if name not in card]
mismatches = {name: (card[name], received[name])
              for name in required if name in card and card[name] != received[name]}
print("빠진 칸", missing)
print("카드와 원문", mismatches)

빠진 칸 []
카드와 원문 {'page': (2, 1)}


빈 목록만 보면 통과할 뻔했습니다. page의 두 숫자 2와 1은 각각 제출된 카드와 원문 페이지이므로 검수자는 카드의 페이지를 고쳐야 합니다.

### 작업을 넘길 때 쓰는 인수 표

<table><thead><tr><th scope="col">맡은 일</th><th scope="col">넘길 것</th><th scope="col">상대방이 확인할 일</th></tr></thead><tbody><tr><td>자료 담당</td><td>rules-v1의 RULE-A 전체 레코드</td><td>원문에서 1쪽 2.1절을 찾는다.</td></tr><tr><td>화면 담당</td><td>Q01의 결과 카드</td><td>카드의 여섯 칸을 같은 판본의 원문 값과 대조한다.</td></tr></tbody></table>

팀에 쪽수가 없는 웹 문서만 있다면 원문 링크와 절 제목처럼 실제로 찾아갈 수 있는 필드로 양쪽 계약을 함께 바꾸세요.

<mark>필드가 있다는 사실과 그 값이 맞다는 사실은 서로 다른 검사를 거칩니다.</mark> card의 page는 정수이며 비어 있지도 않습니다. 원문의 페이지와 대조할 때 비로소 잘못된 위치가 드러납니다.

89장의 자료 목록에 검수 기준이 된 판본과 원문 위치를 적어 두세요.

## 팀 사례에 적용하기 · 시나리오를 검수할 출처 계약 정하기

위 예제에 첫 셀에서 읽어 둔 팀 기록을 넣어 봅니다.

In [ ]:
fields = list(required)
contracts = [{**row, "expected_fields": fields if row["source_id"] else []}
             for row in incoming["scenarios"]]
new = {"contracts": contracts, "fields": fields}
print([(row["id"], len(row["expected_fields"])) for row in contracts])

[('Q01', 6), ('Q02', 0), ('Q03', 0)]


expected_fields가 빈 행에는 이전 답의 카드도 남기지 않습니다. 인정할 위치가 쪽과 절인지, 주소와 문단인지 팀 원문 형식에 맞춰 fields를 고를 수 있습니다.

In [ ]:
output_file = project_root / "team-project/final/chapter88.json"
output_file.parent.mkdir(parents=True, exist_ok=True)
with output_file.open("a+", encoding="utf-8") as handle:
    handle.seek(0)
    previous_text = handle.read()
    if previous_text and json.loads(previous_text) != new:
        raise FileExistsError("같은 장의 다른 결과가 있습니다. 기존 JSON을 다른 이름으로 보관한 뒤 다시 실행하세요.")
    if not previous_text:
        json.dump(new, handle, ensure_ascii=False, indent=2)
        handle.write("\n")
print(output_file.relative_to(project_root).as_posix())

team-project/final/chapter88.json


89장 첫 셀이 이 파일을 읽습니다. 입력을 바꿔 다시 실행할 때는 기존 결과 파일의 이름을 바꿔 보관하세요.

## 돌아보기

다음 자료 담당자에게 CR-02의 여섯 출처 필드를 넘기세요. 원문을 읽어도 채울 수 없는 필드가 있으면 어떤 단계의 일정이 밀리는지 함께 표시합니다.
